# 学習済みの翻訳モデルを自分の分野のデータでファインチューニングする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/27_machine_translation/mt_finetune.ipynb)

NLP クックブック 27-11 ／ 機械翻訳

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

公開されている翻訳モデルを、自分の分野の対訳データで追加学習して、その分野での訳質を上げたい。ここでは、田中コーパス（Tatoeba の日英対訳。辞書や教科書の例文のような短い文）を「自分の分野」に見立てる。

In [1]:
!pip install -q datasets sentencepiece sacremoses sacrebleu accelerate

In [2]:
import gzip
import random
import re
import urllib.request
from datasets import Dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "Helsinki-NLP/opus-mt-ja-en"        # 日英の Marian モデル (約 300MB)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 田中コーパス (EDRDG 配布版、CC BY)。日本語・英語のどちらかが同じ対は 1 つだけ残してから分ける
raw = gzip.decompress(urllib.request.urlopen("https://www.edrdg.org/pub/Nihongo/examples.utf.gz").read()).decode("utf-8")
pairs = [m.groups() for m in re.finditer(r"^A: (.+?)\t(.+?)#ID=", raw, re.M)]
seen_ja, seen_en, uniq = set(), set(), []
for ja, en in sorted(pairs):
    if ja not in seen_ja and en not in seen_en:
        seen_ja.add(ja); seen_en.add(en); uniq.append((ja, en))
random.seed(0)
random.shuffle(uniq)
test = Dataset.from_list([{"ja": ja, "en": en} for ja, en in uniq[:500]])
train = Dataset.from_list([{"ja": ja, "en": en} for ja, en in uniq[1500:31500]])   # 3 万文対だけ使う

def preprocess(batch):
    return tokenizer(batch["ja"], text_target=batch["en"], truncation=True, max_length=64)   # text_target が訳文側 (labels になる)

train_enc = train.map(preprocess, batched=True, remove_columns=["ja", "en"])
test_enc = test.map(preprocess, batched=True, remove_columns=["ja", "en"])

In [3]:
import sacrebleu
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

@torch.no_grad()
def translate(model, sentences, batch_size=64):
    model.to(device).eval()
    out = []
    for i in range(0, len(sentences), batch_size):
        batch = tokenizer(sentences[i:i + batch_size], return_tensors="pt", padding=True, truncation=True, max_length=64).to(device)
        out += tokenizer.batch_decode(model.generate(**batch, num_beams=4, max_length=64), skip_special_tokens=True)
    return out

before = translate(model, test["ja"])
print("微調整前:", sacrebleu.corpus_bleu(before, [test["en"]]))

微調整前: BLEU = 39.87 68.0/47.0/35.2/26.9 (BP = 0.957 ratio = 0.957 hyp_len = 4500 ref_len = 4700)


In [4]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    output_dir="mt-finetuned", num_train_epochs=1, learning_rate=3e-5, warmup_steps=100, weight_decay=0.01,
    per_device_train_batch_size=32, fp16=True, logging_steps=100, save_strategy="no", report_to="none", seed=0,
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_enc, data_collator=DataCollatorForSeq2Seq(tokenizer, model=model))
trainer.train()

Step,Training Loss
100,1.031936
200,0.970650
300,0.988814
400,0.972107
500,0.955763
600,0.987951
700,0.957968
800,1.003132
900,0.954375


TrainOutput(global_step=938, training_loss=0.9795364022000767, metrics={'train_runtime': 39.0855, 'train_samples_per_second': 767.547, 'train_steps_per_second': 23.999, 'total_flos': 206936805998592.0, 'train_loss': 0.9795364022000767, 'epoch': 1.0})

In [5]:
after = translate(model, test["ja"])
print("微調整前:", sacrebleu.corpus_bleu(before, [test["en"]]))
print("微調整後:", sacrebleu.corpus_bleu(after, [test["en"]]))
for ja, en, b, a in list(zip(test["ja"], test["en"], before, after))[:6]:
    print(f"{ja}\n   前: {b}\n   後: {a}\n   参照: {en}")

model.save_pretrained("mt-finetuned"); tokenizer.save_pretrained("mt-finetuned")

微調整前: BLEU = 39.87 68.0/47.0/35.2/26.9 (BP = 0.957 ratio = 0.957 hyp_len = 4500 ref_len = 4700)
微調整後: BLEU = 43.75 69.9/49.8/38.2/29.4 (BP = 0.984 ratio = 0.984 hyp_len = 4626 ref_len = 4700)
そんなことをするなんて何と無分別な男だ。
   前: What a senseless man to do such a thing!
   後: What a foolish man to do such a thing!
   参照: What a thoughtless man to do that!
我々が手に手を取って一致団結すれば、きっとこの不況の時代も乗り切れるだろう。
   前: If we put our hands in our hands and united, we'll probably be able to survive this recession.
   後: If we put our hands in our hands and unite, we will probably be able to survive this period of depression.
   参照: If we hang tight, I'm sure we can make it through these bad times.
それは夢のように思われます。
   前: It seems like a dream.
   後: It seems like a dream.
   参照: It sounds like a dream.
話し合わねばならないことがたくさんある。
   前: There's a lot to talk about.
   後: We have a lot to discuss.
   参照: We have a lot of ground to cover.
私はそのことについて以前に聞いたことがある。
   前: I've heard about it before.
   後: I have heard about it before.


- ゼロから学習するよりはるかに少ないデータと時間で、特定の分野の文体や用語に合わせられる。検証の GPU で 3 万文対・1 エポックが 1 分弱（T4 ならその 2〜3 倍）。
- 実測の BLEU は 39.9 → 43.8。「I've」→「I have」のように短縮形を避けるなど、田中コーパスの英訳の文体に寄る。2 エポックにしても、全 13.8 万文対で学習しても 43.3〜43.6 で、それ以上は上がらなかった。
- 微調整前から BLEU が高いのは、`opus-mt-ja-en` が Tatoeba を含む OPUS のデータで学習されていて、テスト文をすでに見ている可能性があるため。分野が元のモデルの学習データと重なっていると、微調整の効果は出にくい（字幕が多い opus-100 で同じことをすると、BLEU は 17.6 → 17.5 で上がらなかった）。微調整の前に、元のモデルがその分野をどれだけ訳せるかを必ず測る。
- 学習率は小さめ (1e-5〜5e-5) にする。大きすぎると、元のモデルが持っていた一般的な翻訳能力が壊れる（破滅的忘却）。
- 微調整後は、対象の分野だけでなく、一般的な文のテストセットでも評価する。分野内で上がって分野外で下がるのが典型的な副作用。
- BLEU の差が小さいときは、評価指標と統計的検定の章の検定（ブートストラップ）で有意かどうかを確かめる。訳文を実際に見比べることも欠かせない。
- NLLB のような多言語モデルや、LLM を LoRA で微調整する方法（LLM の学習と調整の章の「ベースモデルを LoRA で指示チューニングする (SFT)」）でも同じ考え方が使える。